<a href="https://colab.research.google.com/github/Willian-devprog/Colecao_de.Filmes/blob/main/Colecao_de.Filmes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Coleção de Filmes e Séries — Sistema de Gerenciamento e Avaliação

Este projeto consiste em uma aplicação em **Python** desenvolvida para execução no **Google Colab/VS Code**, criada como projeto prático para a disciplina de **Programação I** da UFRPE. O sistema permite que usuários gerenciem suas coleções de filmes e séries, registrem avaliações com comentários e analisem seus hábitos de consumo de mídias sem a necessidade de um banco de dados externo, utilizando persistência em arquivos **CSV**.

## 👥 Integrantes do Grupo e Responsabilidades
| Nome Completo | Responsabilidade no Projeto |
| :--- | :--- |
| **José Bezerra Cabral Neto** | Desenvolvimento das rotinas de autenticação (Login/Cadastro) e controle de permissões. |
| **Marcos A. Lira Gomes** | Implementação das funções de cadastro de mídias, importação CSV e filtro por palavra-chave. |
| **Willian Hassan Costa Santos** | Criação do sistema de avaliação (0-10 com comentários), ranking, recomendações e exportação de relatórios. |

## 🚀 Funcionalidades do Sistema
- **Autenticação e Controle de Acesso:**
  - **Consulta Aberta**: Qualquer pessoa pode buscar e visualizar mídias e avaliações sem precisar fazer login.
  - **Área do Usuário**: Para cadastrar novas mídias ou registrar avaliações, é obrigatório efetuar login no sistema.
- **Cadastro de Mídias (Filmes e Séries):**
  - Registro individual informando título, categoria/gênero, tipo (Filme ou Série) e ano de lançamento.
- **Importação de Coleção Pessoal (CSV):**
  - Permite importar de forma automatizada um arquivo `.csv` contendo uma lista prévia de filmes e séries.
- **Avaliações com Comentários:**
  - Atribuição de notas em um intervalo fechado de **0.0 a 10.0** (onde `0` representa *Não recomendaria* e `10` representa *Super recomendaria / Perfeito*).
  - Espaço dedicado para inclusão de comentários e opiniões pessoais sobre a mídia.
- **Busca por Palavra-Chave:**
  - Pesquisa rápida de mídias por título, gênero ou tipo.
- **Ranking Geral:**
  - Listagem das mídias mais bem avaliadas pelos usuários organizadas por nota.
- **Recomendações Personalizadas:**
  - Algoritmo simples de recomendação baseado nos gêneros mais consumidos e bem avaliados pelo usuário logado.
- **Exportação de Relatórios:**
  - Geração de relatórios sumarizados em arquivo de texto (`.txt`) com estatísticas da coleção e histórico de avaliações.

## 🛠️ Tecnologias e Bibliotecas Utilizadas
- **Linguagem:** Python 3
- **Ambiente de Desenvolvimento:** Visual Studio Code (VS Code)
- **Ambiente de Execução Alternativo:** Google Colab
- **Persistência de Dados:** Arquivos `.csv` e `.txt` (sem uso de banco de dados relacional/SQL)
- **Bibliotecas Nativas do Python:**
  - `csv`: Para leitura e escrita estruturada dos dados da coleção e usuários.
  - `os` / `sys`: Para manipulação e verificação de arquivos e caminhos no ambiente Colab.
  
## 📋 Instruções de Execução Passo a Passo (Google Colab)
1. **Acessar o Notebook:**
   - Abra o arquivo `.ipynb` do projeto diretamente no Google Colab.
2. **Preparar os Arquivos de Dados:**
   - Certifique-se de carregar os arquivos `.csv` necessários na aba **Arquivos** (ícone de pasta no menu esquerdo do Colab) ou execute a célula inicial do código para que o programa crie automaticamente a estrutura de dados inicial.
3. **Executar a Aplicação:**
   - Execute a célula principal do projeto para iniciar o menu interativo via terminal/console do Colab.
4. **Navegação:**
   - Utilize as opções numéricas do menu para navegar entre as rotinas de consulta, login, cadastro, avaliação e exportação de relatórios.

## 📋 Instruções de Execução Passo a Passo (VS Code)
1. **Clonar o Repositório:**
``bash
    git clone https://github.com/Willian-devprog/Colecao_de.Filmes.git
    cd Colecao_de.Filmes

## 📂 Estrutura dos Arquivos de Dados (CSV)
O projeto armazena as informações em arquivos texto com separador `;` para garantir a persistência dos dados entre execuções:
   - `usuarios.csv`: Armazena os dados dos usuários cadastrados (`id;nome;login;senha`).
   - `midias.csv`: Contém o acervo de filmes e séries (`id;titulo;tipo;genero;ano`).
   - `avaliacoes.csv`: Registra o histórico de avaliações (`id_midia;id_usuario;nota;comentario`)
  


In [3]:

import csv
import json
from datetime import datetime
from pathlib import Path

ARQUIVO = Path("filmes.json")


def carregar():
    # Lê os filmes salvos e adapta registros da versão antiga.
    if not ARQUIVO.exists():
        return []

    try:
        with open(ARQUIVO, "r", encoding="utf-8") as f:
            filmes = json.load(f)

        if not isinstance(filmes, list):
            print("Formato inválido no arquivo JSON. Iniciando com lista vazia.")
            return []

        alterado = False

        for filme in filmes:
            # Compatibilidade com a versão antiga:
            # categoria passa a ser genero.
            if "genero" not in filme and "categoria" in filme:
                filme["genero"] = filme["categoria"]
                alterado = True

            if "categoria" in filme:
                del filme["categoria"]
                alterado = True

            # Registros antigos são considerados filmes.
            if "tipo" not in filme:
                filme["tipo"] = "Filme"
                alterado = True

        # Salva automaticamente os dados adaptados.
        if alterado:
            salvar(filmes)

        return filmes

    except json.JSONDecodeError:
        print("Arquivo corrompido. Iniciando com lista vazia.")
        return []

    except (OSError, TypeError, AttributeError):
        print("Erro ao carregar os dados. Iniciando com lista vazia.")
        return []


def salvar(filmes):
    # Grava a lista inteira no arquivo JSON.
    with open(ARQUIVO, "w", encoding="utf-8") as f:
        json.dump(filmes, f, ensure_ascii=False, indent=2)


def validar_ano(texto):
    # Converte o ano para inteiro e confere se é um valor razoável.
    try:
        ano = int(texto)
    except ValueError:
        raise ValueError("O ano deve ser um número inteiro.")

    ano_maximo = datetime.now().year + 5

    if not 1888 <= ano <= ano_maximo:
        raise ValueError(f"O ano deve estar entre 1888 e {ano_maximo}.")

    return ano


def validar_nota(texto):
    # Converte a nota para decimal entre 0.0 e 10.0.
    try:
        nota = float(texto.replace(",", "."))
    except ValueError:
        raise ValueError("A nota deve ser um número (ex: 8 ou 7.5).")

    if not 0.0 <= nota <= 10.0:
        raise ValueError("A nota deve estar entre 0.0 e 10.0.")

    return round(nota, 1)


def criar_filme(titulo, genero, tipo, ano, nota, comentario=""):
    # Valida os campos e devolve o dicionário da mídia.
    titulo = titulo.strip()
    genero = genero.strip()
    tipo = tipo.strip().capitalize()
    ano = str(ano).strip()
    nota = str(nota).strip()

    if not titulo or not genero or not tipo or not ano or not nota:
        raise ValueError(
            "Título, gênero, tipo, ano e nota são obrigatórios."
        )

    return {
        "titulo": titulo,
        "genero": genero,
        "tipo": tipo,
        "ano": validar_ano(ano),
        "nota": validar_nota(nota),
        "comentario": comentario.strip(),
    }


def filme_existe(filmes, titulo):
    # Verifica se já existe uma mídia com esse título.
    return any(
        f["titulo"].lower() == titulo.strip().lower()
        for f in filmes
    )


def exibir(lista):
    # Mostra as mídias na tela.
    for i, f in enumerate(lista, 1):
        comentario = f.get("comentario", "")

        print(
            f"{i}. [{f['tipo']}] {f['titulo']} "
            f"({f['ano']}) - Gênero: {f['genero']} "
            f"- Nota: {f['nota']:.1f}"
        )

        if comentario:
            print(f"   Comentário: {comentario}")


def adicionar(filmes):
    # Opção 1: cadastra uma nova mídia.
    print("\n--- Adicionar nova mídia ---")

    titulo = input("Título: ").strip()

    if titulo and filme_existe(filmes, titulo):
        print(f"Atenção: '{titulo}' já consta na sua coleção!")
        return

    print("\nTipos de mídia:")
    print("1. Filme")
    print("2. Série")
    print("3. Documentário")
    print("4. Animação")

    opcao_tipo = input("Escolha o tipo de mídia (1-4): ").strip()

    tipos = {
        "1": "Filme",
        "2": "Série",
        "3": "Documentário",
        "4": "Animação"
    }

    tipo = tipos.get(opcao_tipo)

    if not tipo:
        print("Tipo de mídia inválido.")
        return

    genero = input("Gênero (ex: Ação, Comédia, Terror): ")
    ano = input("Ano de lançamento: ")
    nota = input("Nota (0.0 a 10.0): ")
    comentario = input("Comentário (opcional): ")

    try:
        filme = criar_filme(
            titulo, genero, tipo, ano, nota, comentario
        )

    except ValueError as erro:
        print(f"Erro: {erro}")
        return

    filmes.append(filme)
    salvar(filmes)

    print(f"Mídia '{filme['titulo']}' adicionada com sucesso!")


def listar(filmes):
    # Opção 2: mostra todas as mídias.
    if not filmes:
        print("\nSua coleção está vazia.")
        return

    print(f"\n--- Sua Coleção ({len(filmes)} mídias) ---")
    exibir(filmes)


def ordenar(filmes):
    # Opção 3: ordena a coleção por título, ano ou nota.
    if not filmes:
        print("\nNenhuma mídia cadastrada para ordenar.")
        return

    print("\n--- Critérios de Ordenação ---")
    print("1. Por Título (A-Z)")
    print("2. Por Ano (mais antigo ao mais recente)")
    print("3. Por Nota (maior para menor)")

    criterio = input("Escolha o critério (1-3): ").strip()

    if criterio == "1":
        filmes.sort(key=lambda f: f["titulo"].lower())

    elif criterio == "2":
        filmes.sort(key=lambda f: f["ano"])

    elif criterio == "3":
        filmes.sort(key=lambda f: f["nota"], reverse=True)

    else:
        print("Opção de ordenação inválida.")
        return

    salvar(filmes)
    print("Mídias ordenadas com sucesso!")


def buscar(filmes):
    # Opção 4: pesquisa por título, gênero ou tipo.
    if not filmes:
        print("\nNenhuma mídia cadastrada para buscar.")
        return

    palavra = input("\nDigite a palavra-chave: ").strip().lower()

    if not palavra:
        print("Digite pelo menos uma letra para buscar.")
        return

    encontrados = [
        f for f in filmes
        if palavra in f["titulo"].lower()
        or palavra in f["genero"].lower()
        or palavra in f["tipo"].lower()
    ]

    if not encontrados:
        print(f"Nenhuma mídia encontrada para '{palavra}'.")
        return

    print(f"\n--- {len(encontrados)} resultado(s) para '{palavra}' ---")
    exibir(encontrados)


def importar_csv(filmes):
    # Opção 5: importa mídias de um arquivo CSV.
    print("\n--- Importar CSV ---")
    print(
        "Colunas esperadas: titulo, genero, tipo, ano, nota, "
        "comentario (opcional)"
    )

    caminho = Path(input("Caminho do arquivo .csv: ").strip().strip('"'))

    if not caminho.is_file():
        print("Erro: arquivo não encontrado.")
        return

    importados = 0
    duplicados = 0
    erros = []

    try:
        with open(
            caminho, "r", encoding="utf-8-sig", newline=""
        ) as f:

            # Identifica o separador do CSV.
            cabecalho = f.readline()
            separador = (
                ";" if cabecalho.count(";") > cabecalho.count(",")
                else ","
            )
            f.seek(0)

            leitor = csv.DictReader(f, delimiter=separador)

            # Normaliza os nomes das colunas.
            if leitor.fieldnames:
                leitor.fieldnames = [
                    c.strip().lower() for c in leitor.fieldnames
                ]

            obrigatorias = {"titulo", "ano", "nota"}

            # Aceita genero ou categoria para compatibilidade.
            if not leitor.fieldnames or not obrigatorias.issubset(
                leitor.fieldnames
            ) or not ({"genero", "categoria"} & set(leitor.fieldnames)):
                print(
                    "Erro: o cabeçalho deve conter titulo, genero "
                    "(ou categoria), ano e nota."
                )
                return

            # Os dados começam na linha 2.
            for numero_linha, linha in enumerate(leitor, start=2):

                titulo = (linha.get("titulo") or "").strip()

                if titulo and filme_existe(filmes, titulo):
                    duplicados += 1
                    continue

                genero = (
                    linha.get("genero")
                    or linha.get("categoria")
                    or ""
                )

                # CSV antigo sem tipo será importado como Filme.
                tipo = linha.get("tipo") or "Filme"

                try:
                    filme = criar_filme(
                        titulo,
                        genero,
                        tipo,
                        linha.get("ano") or "",
                        linha.get("nota") or "",
                        linha.get("comentario") or "",
                    )

                except ValueError as erro:
                    erros.append(f"Linha {numero_linha}: {erro}")
                    continue

                filmes.append(filme)
                importados += 1

    except (OSError, csv.Error, UnicodeError) as erro:
        print(f"Erro ao ler o arquivo CSV: {erro}")
        return

    if importados:
        salvar(filmes)

    print("\n--- Resultado da importação ---")
    print(f"Importados: {importados}")
    print(f"Ignorados por já existirem: {duplicados}")
    print(f"Com erro: {len(erros)}")

    for mensagem in erros:
        print(f"  - {mensagem}")


# MENU PRINCIPAL

def gerenciar_filmes():
    filmes = carregar()

    while True:
        print("\n GERENCIADOR DE COLEÇÃO DE MÍDIAS ")
        print("1. Adicionar nova mídia")
        print("2. Ver lista de mídias")
        print("3. Ordenar mídias")
        print("4. Buscar por palavra-chave")
        print("5. Importar de CSV")
        print("6. Sair")

        opcao = input("\nEscolha uma opção (1-6): ").strip()

        if opcao == "1":
            adicionar(filmes)

        elif opcao == "2":
            listar(filmes)

        elif opcao == "3":
            ordenar(filmes)

        elif opcao == "4":
            buscar(filmes)

        elif opcao == "5":
            importar_csv(filmes)

        elif opcao == "6":
            print("Até a próxima!")
            break

        else:
            print("Opção inválida. Escolha de 1 a 6.")


if __name__ == "__main__":
    gerenciar_filmes()



 GERENCIADOR DE COLEÇÃO DE MÍDIAS 
1. Adicionar nova mídia
2. Ver lista de mídias
3. Ordenar mídias
4. Buscar por palavra-chave
5. Importar de CSV
6. Sair

Escolha uma opção (1-6): 2

--- Sua Coleção (1 mídias) ---
1. [Filme] Jogos Vorazes: Amanhecer Na Colheita (2026) - Gênero: Ação, aventura, drama - Nota: 7.6
   Comentário: Parte da crítica e do público aponta trechos arrastados ou, em outros momentos, desenvolvimento apressado das relações e da longa metragem

 GERENCIADOR DE COLEÇÃO DE MÍDIAS 
1. Adicionar nova mídia
2. Ver lista de mídias
3. Ordenar mídias
4. Buscar por palavra-chave
5. Importar de CSV
6. Sair

Escolha uma opção (1-6): 6
Até a próxima!
